# MyDream 확장 시퀀스 모델 비교

Colab에서 현재 GRU 이후의 아키텍처를 비교하고, 알람 구간 결과를 표와 그래프로 확인합니다.

이 노트북은 현재 선택된 GRU와 TCN, Transformer, CNN+GRU를 비교합니다. 또한 분석기의 기본 임계값 목록에는 없지만 배포와 관련된 `0.55` 임계값으로도 비교를 다시 수행합니다.

## 1. 런타임 설정

가능하면 GPU가 있는 Colab 런타임을 사용합니다. Colab 런타임은 서로 분리되어 있으므로 입력 데이터와 생성 결과는 Google Drive의 `PROFILE_ROOT` 아래에 저장합니다.

In [ ]:
!pip -q install tensorflow pandas matplotlib seaborn

## 2. Git 코드 폴더 불러오기

새 Colab 런타임에서는 프로젝트를 복제합니다. 코드 폴더가 이미 있으면 기존 폴더를 그대로 사용합니다.

In [ ]:
from pathlib import Path
import subprocess

CODE_ROOT = Path('/content/mydream-training-evaluation')
REPOSITORY_URL = 'https://github.com/sfpahsdev-mydream/mydream-training-evaluation.git'

if not (CODE_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(CODE_ROOT)], check=True)

assert (CODE_ROOT / 'run_sequence_experiment_matrix.py').exists(), CODE_ROOT
print('코드 폴더 준비 완료:', CODE_ROOT)

## 3. Google Drive 연결 및 입력 확인

이전 런타임에서 만든 시퀀스 데이터와 GRU 결과를 계속 사용하려면 Google Drive의 같은 프로필 폴더에 저장되어 있어야 합니다. 같은 프로필에 `model_tabular_tflite/alarm_predictions_long.csv`가 존재할 때만 `INCLUDE_TABULAR = True`로 설정합니다.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROFILE_ROOT = Path('/content/drive/MyDrive/mydream_latest/out/latest_fixed_wake_policy')
INCLUDE_TABULAR = False
THRESHOLDS = [0.4, 0.5, 0.55, 0.6]

SELECTED_GRU_DIR = PROFILE_ROOT / 'sequence_experiments' / 'gru' / 'gru64_dense32_dropout00'

required_paths = [
    PROFILE_ROOT / 'sequence_60m',
    PROFILE_ROOT / 'sequence_60m_alarm',
    SELECTED_GRU_DIR / 'alarm_predictions_long.csv',
]
for path in required_paths:
    assert path.exists(), f'필수 입력이 없습니다: {path}'

tabular_predictions = PROFILE_ROOT / 'model_tabular_tflite' / 'alarm_predictions_long.csv'
if INCLUDE_TABULAR:
    assert tabular_predictions.exists(), f'테이블 모델 예측 결과가 없습니다: {tabular_predictions}'

print('프로필 루트:', PROFILE_ROOT)
print('기준 GRU 결과 폴더:', SELECTED_GRU_DIR)
print('테이블 모델 포함 여부:', INCLUDE_TABULAR)
print('임계값:', THRESHOLDS)

## 4. 확장 후보 모델 학습

TCN, Transformer, CNN+GRU를 학습한 뒤 스크립트의 표준 비교를 수행합니다. 이미 생성된 후보 예측 결과는 재사용합니다.

In [ ]:
import subprocess
import sys

matrix_command = [
    sys.executable,
    str(CODE_ROOT / 'run_sequence_experiment_matrix.py'),
    '--profile-root', str(PROFILE_ROOT),
    '--experiment-set', 'expanded',
    '--skip-existing',
]
if not INCLUDE_TABULAR:
    matrix_command.append('--no-tabular-model')

print('실행 명령:', ' '.join(matrix_command))
subprocess.run(matrix_command, cwd=CODE_ROOT, check=True)

## 5. 비교 가능한 임계값 결과 다시 계산

현재 운영 후보는 임계값 `0.55`를 사용하므로, 이 셀은 모든 아키텍처에 대해 `0.4`, `0.5`, `0.55`, `0.6`을 사용한 간결한 비교 결과를 작성합니다.

In [ ]:
comparison_root = PROFILE_ROOT / 'sequence_experiments' / 'expanded'
comparison_output = comparison_root / 'alarm_failure_comparison_selected_thresholds'

model_dirs = [
    SELECTED_GRU_DIR,
    comparison_root / 'tcn64_dense32_dropout00',
    comparison_root / 'transformer64_dense32_dropout10',
    comparison_root / 'cnn32_gru64_dense32_dropout00',
]
if INCLUDE_TABULAR:
    model_dirs.insert(0, PROFILE_ROOT / 'model_tabular_tflite')

for model_dir in model_dirs:
    predictions = model_dir / 'alarm_predictions_long.csv'
    assert predictions.exists(), f'모델 예측 결과가 없습니다: {predictions}'

analysis_command = [sys.executable, str(CODE_ROOT / 'analyze_alarm_failures.py')]
for model_dir in model_dirs:
    analysis_command += ['--model-dir', str(model_dir)]
for threshold in THRESHOLDS:
    analysis_command += ['--threshold', str(threshold), '--focus-threshold', str(threshold)]
analysis_command += ['--output-dir', str(comparison_output)]

print('실행 명령:', ' '.join(analysis_command))
subprocess.run(analysis_command, cwd=CODE_ROOT, check=True)
print('비교 결과 폴더:', comparison_output)

## 6. 임계값 비교 결과 보기

`strong_fail`은 낮게 유지하면서 `deep_success`와 `success_per_smart`가 높은 모델을 우선합니다. 현재 배포 기준 임계값은 `0.55`입니다.

In [ ]:
import pandas as pd
from IPython.display import display

summary_path = comparison_output / 'model_comparison_summary.csv'
summary = pd.read_csv(summary_path)
summary = summary.sort_values(['threshold', 'strong_fail', 'deep_success'], ascending=[True, True, False])

display(summary.reset_index(drop=True))

reference = summary[summary['threshold'].eq(0.55)].copy()
reference = reference.sort_values(['strong_fail', 'deep_success', 'success_per_smart'], ascending=[True, False, False])
print('임계값 0.55 비교')
display(reference.reset_index(drop=True))

## 7. 시각적 비교

첫 번째 차트에서 `deep_success`와 `strong_fail` 사이의 균형을 판단합니다. GRU보다 실질적으로 나은 후보라면 같은 임계값에서 실패는 줄고 성공은 늘어나는 방향이어야 합니다.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.scatterplot(
    data=summary,
    x='strong_fail',
    y='deep_success',
    hue='model',
    style='threshold',
    s=120,
    ax=axes[0],
)
axes[0].set_title('임계값별 알람 성능 균형')
axes[0].set_xlabel('강한 실패 (낮을수록 좋음)')
axes[0].set_ylabel('깊은 수면 성공 (높을수록 좋음)')

plot_data = reference.melt(
    id_vars=['model', 'threshold'],
    value_vars=['deep_success', 'strong_fail'],
    var_name='metric',
    value_name='count',
)
sns.barplot(data=plot_data, x='model', y='count', hue='metric', errorbar=None, ax=axes[1])
axes[1].set_title('임계값 0.55에서 모델별 개수')
axes[1].set_xlabel('모델')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 8. 비교 파일 내보내기

Colab 런타임이 임시 환경이면 간결한 비교 결과를 다운로드합니다. 생성된 모델 산출물과 데이터셋은 Git 외부에 유지해야 합니다.

In [ ]:
import shutil

archive_path = shutil.make_archive(
    str(comparison_output),
    'zip',
    root_dir=comparison_output,
)
print('생성 완료:', archive_path)

# Colab에서 결과 요약 압축 파일을 다운로드하려면 아래 주석을 해제합니다.
# from google.colab import files
# files.download(archive_path)